In [ ]:
import requests
import pandas as pd
OVERPASS_URL = "https://overpass-api.de/api/interpreter"

In [ ]:
# using a names area
# Use area when you want to search within:
# 
# City/county/state boundaries
# Official administrative regions
# Don't need exact coordinates


query = """
[out:json][timeout:60];

area
  ["name"="Chicago"]
  ["boundary"="administrative"]
  ["admin_level"="8"]
  ->.searchArea;

(
  node
    ["amenity"="restaurant"]
    ["diet:gluten_free"="yes"]
    (area.searchArea);

  way
    ["amenity"="restaurant"]
    ["diet:gluten_free"="yes"]
    (area.searchArea);

  relation
    ["amenity"="restaurant"]
    ["diet:gluten_free"="yes"]
    (area.searchArea);
);

out center tags;
"""


In [ ]:
# using around instead of area


query = """
[out:json][timeout:60];

(
  node
    ["amenity"="restaurant"]
    ["diet:gluten_free"="yes"]
    (around:5000, 41.8781, -87.6298);  // 5km radius around Chicago center
  
  way
    ["amenity"="restaurant"]
    ["diet:gluten_free"="yes"]
    (around:5000, 41.8781, -87.6298);
  
  relation
    ["amenity"="restaurant"]
    ["diet:gluten_free"="yes"]
    (around:5000, 41.8781, -87.6298);
);

out center tags;
"""

Common diet tags:

diet:vegan - Serves vegan options
diet:vegetarian - Serves vegetarian options
diet:gluten_free - Serves gluten-free options (what you're currently using)
diet:lactose_free - Lactose-free options
diet:halal - Halal food
diet:kosher - Kosher food
diet:pescetarian - Pescetarian options (fish but no meat)


Less common but valid:

diet:organic - Organic food
diet:raw - Raw food diet
diet:fruitarian - Fruitarian diet
diet:low_fat - Low-fat options
diet:sugar_free - Sugar-free options
diet:dairy_free - Dairy-free options


Values:
Each tag typically uses:

yes - Available
no - Not available
only - Only serves this type (e.g., diet:vegan=only for vegan-only restaurants)
limited - Limited availability

 more options:

amenity"~"restaurant|cafe|fast_food|food_court|bar|pub

Search for ANY of these diets
["diet:vegan"="yes"]
["diet:vegetarian"="yes"]
["diet:gluten_free"="yes"]

Or use regex for any diet option
["~diet:.*"~"yes|only"]

"takeaway"="yes" - Takeout available
"delivery"="yes" - Delivery service
"internet_access"="wlan" - WiFi available
"outdoor_seating"="yes" - Outdoor dining
"wheelchair"="yes" - Accessibility
"payment:cash"="yes" - Payment methods
"payment:credit_cards"="yes"

"opening_hours"  # Returns hours like "Mo-Su 11:00-22:00"

["cuisine"~"pizza|mexican|chinese|thai|indian"]

In [ ]:
query = """
[out:json][timeout:60];

(
  node
    ["amenity"~"restaurant|cafe|fast_food"]
    ["diet:vegetarian"="yes"]
    ["takeaway"="yes"]
    (around:5000, 41.8781, -87.6298);
  
  way
    ["amenity"~"restaurant|cafe|fast_food"]
    ["diet:vegetarian"="yes"]
    ["takeaway"="yes"]
    (around:5000, 41.8781, -87.6298);
  
  relation
    ["amenity"~"restaurant|cafe|fast_food"]
    ["diet:vegetarian"="yes"]
    ["takeaway"="yes"]
    (around:5000, 41.8781, -87.6298);
);

out center tags;
"""

In [ ]:
import requests
headers = {
    "User-Agent": "Python Overpass API Client"
}

response = requests.get(
    OVERPASS_URL,
    params={"data": query},
    headers=headers,
    timeout=90
)
response.raise_for_status()
data = response.json()
print(data)

In [ ]:
restaurants = []

for element in data["elements"]:
    tags = element.get("tags", {})

    # Nodes have lat/lon directly.
    # Ways and relations usually return a "center" when using `out center tags`.
    if element["type"] == "node":
        lat = element.get("lat")
        lon = element.get("lon")
    else:
        center = element.get("center", {})
        lat = center.get("lat")
        lon = center.get("lon")

    restaurants.append({
        "osm_type": element["type"],
        "osm_id": element["id"],
        "name": tags.get("name"),
        "cuisine": tags.get("cuisine"),
        "diet_gluten_free": tags.get("diet:gluten_free"),
        "website": tags.get("website"),
        "phone": tags.get("phone") or tags.get("contact:phone"),
        "street": tags.get("addr:street"),
        "housenumber": tags.get("addr:housenumber"),
        "city": tags.get("addr:city"),
        "postcode": tags.get("addr:postcode"),
        "lat": lat,
        "lon": lon
    })

print(restaurants[0])

In [ ]:

df = pd.DataFrame(restaurants)
print(df.head(20))

# Optional: save results
df.to_csv("chicago_gluten_free_restaurants_osm.csv", index=False)